In [1]:
import subprocess
subprocess.run(["pip", "install", "azure-storage-blob", "pyarrow", "pandas", "-q"])

CompletedProcess(args=['pip', 'install', 'azure-storage-blob', 'pyarrow', 'pandas', '-q'], returncode=0)

In [2]:
import os
import io
import sys
import pandas as pd
from pyspark.sql import SparkSession
from azure.storage.blob import BlobServiceClient
 
ACCOUNT_NAME = os.environ["AZURE_STORAGE_ACCOUNT_NAME"]
ACCOUNT_KEY  = os.environ["AZURE_STORAGE_ACCOUNT_KEY"]
CONTAINER    = os.environ.get("AZURE_CONTAINER_NAME", "nyc-traffic-lakehouse")
CONN_STR     = os.environ["AZURE_STORAGE_CONNECTION_STRING"]
 
spark = (SparkSession.builder
    .appName("Fix_Artifacts")
    .master("spark://spark-master:7077")
    .config("spark.sql.session.timeZone", "America/New_York")
    .config("spark.pyspark.python", "/usr/bin/python3")
    .config("spark.executorEnv.PYSPARK_PYTHON", "/usr/bin/python3")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-azure:3.3.4")
    .config(f"spark.hadoop.fs.azure.account.key.{ACCOUNT_NAME}.blob.core.windows.net", ACCOUNT_KEY)
    .getOrCreate())
 
BASE_PATH = f"wasbs://{CONTAINER}@{ACCOUNT_NAME}.blob.core.windows.net"
blob_service = BlobServiceClient.from_connection_string(CONN_STR)
container_client = blob_service.get_container_client(CONTAINER)
 
print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"BASE_PATH: {BASE_PATH}")

Spark version: 4.0.0
Master: spark://spark-master:7077
BASE_PATH: wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net


In [3]:
import pyspark
print(f"PySpark version (Jupyter driver): {pyspark.__version__}")
print(f"Spark version (cluster): {spark.version}")
print(f"Python version: {sys.version}")

PySpark version (Jupyter driver): 4.0.0
Spark version (cluster): 4.0.0
Python version: 3.13.5 | packaged by conda-forge | (main, Jun 16 2025, 08:27:50) [GCC 13.3.0]


In [4]:
prefix = "artifacts/model/random_forest_model/"
blobs = list(container_client.list_blobs(name_starts_with=prefix))
for b in blobs:
    print(b.name)

artifacts/model/random_forest_model/metadata/_SUCCESS
artifacts/model/random_forest_model/metadata/_committed_7922765101422405844
artifacts/model/random_forest_model/metadata/_started_7922765101422405844
artifacts/model/random_forest_model/metadata/part-00000-tid-7922765101422405844-c83ae4e9-1df1-490b-8a9d-46f5d8988537-714-1-c000.txt
artifacts/model/random_forest_model/stages/0_StringIndexer_3d4985d06cfa/data/_SUCCESS
artifacts/model/random_forest_model/stages/0_StringIndexer_3d4985d06cfa/data/_committed_7914965100815917641
artifacts/model/random_forest_model/stages/0_StringIndexer_3d4985d06cfa/data/_started_7914965100815917641
artifacts/model/random_forest_model/stages/0_StringIndexer_3d4985d06cfa/data/part-00000-tid-7914965100815917641-21b3f530-c7e0-4892-b149-e60c862b7ba2-716-1.c000.snappy.parquet
artifacts/model/random_forest_model/stages/0_StringIndexer_3d4985d06cfa/metadata/_SUCCESS
artifacts/model/random_forest_model/stages/0_StringIndexer_3d4985d06cfa/metadata/_committed_4038691

In [5]:
from pyspark.ml import PipelineModel

MODEL_PATH = f"{BASE_PATH}/artifacts/model/random_forest_model"
try:
    model = PipelineModel.load(MODEL_PATH)
    print("Load thành công")
except Exception as e:
    print("Load thất bại:")
    print(repr(e))

{"ts": "2026-06-30 15:06:32.363", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `treeID` cannot be resolved. Did you mean one of the following? [`_1`, `_2`, `_3`]. SQLSTATE: 42703", "context": {"file": "org.apache.spark.ml.tree.EnsembleModelReadWrite$.loadImpl(treeModels.scala", "line": "525)", "fragment": "select", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o118.load.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `treeID` cannot be resolved. Did you mean one of the following? [`_1`, `_2`, `_3`]. SQLSTATE: 42703;\n'Project ['treeID, 'metadata, 'weights]\n+- Relation [_1#40,_2#41,_3#42] parquet\n\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.unresolvedAttributeError(QueryCompilationErrors.scala:40

Load thất bại:
AnalysisException()


In [6]:
import pyarrow.parquet as pq
import pyarrow as pa

RF_STAGE = "3_RandomForestClassifier_b44ed63ef3d2"
TREES_META_PREFIX = f"artifacts/model/random_forest_model/stages/{RF_STAGE}/treesMetadata/"

# Tìm đúng file part-* (bỏ qua _SUCCESS, _committed, _started)
blobs = list(container_client.list_blobs(name_starts_with=TREES_META_PREFIX))
parquet_blob_name = [b.name for b in blobs if b.name.endswith(".parquet")][0]
print("File cần sửa:", parquet_blob_name)

local_path = "/tmp/treesMetadata_original.parquet"
blob_client = container_client.get_blob_client(parquet_blob_name)
with open(local_path, "wb") as f:
    f.write(blob_client.download_blob().readall())

table = pq.read_table(local_path)
print("Schema gốc:", table.schema)
print("Số rows:", table.num_rows)

File cần sửa: artifacts/model/random_forest_model/stages/3_RandomForestClassifier_b44ed63ef3d2/treesMetadata/part-00000-tid-4148136318540938165-0e677ae4-154e-4e4b-8b1b-2aeb2b8b885e-729-1.c000.snappy.parquet
Schema gốc: _1: int32 not null
_2: string
_3: double not null
-- schema metadata --
org.apache.spark.version: '4.1.0'
org.apache.spark.sql.parquet.row.metadata: '{"type":"struct","fields":[{"' + 182
com.databricks.spark.jobGroupId: '1001782823154939_4826137363939899901_44' + 30
com.databricks.spark.clusterId: '0630-121047-2aj8xbbf-v2n'
com.databricks.spark.sparkImageLabel: 'release__18.2.x-snapshot-photon-sc' + 73
com.databricks.spark.writeTimestamp: '2026-06-30T13:21:51.644721094Z'
com.databricks.spark.parquetFormatVersion: '1.0.0'
Số rows: 80


In [7]:
new_table = table.rename_columns(["treeID", "metadata", "weights"])
print("Schema mới:", new_table.schema)

fixed_local_path = "/tmp/treesMetadata_fixed.parquet"
pq.write_table(new_table, fixed_local_path)

Schema mới: treeID: int32 not null
metadata: string
weights: double not null


In [9]:
import pandas as pd

df_check = pd.read_parquet(fixed_local_path)
print(df_check.dtypes)
print(df_check.shape)
df_check.head()

treeID        int32
metadata     object
weights     float64
dtype: object
(80, 3)


,treeID,metadata,weights
0,0,"{""class"":""org.apache.spark.ml.classification.D...",1.0
1,1,"{""class"":""org.apache.spark.ml.classification.D...",1.0
2,2,"{""class"":""org.apache.spark.ml.classification.D...",1.0
3,3,"{""class"":""org.apache.spark.ml.classification.D...",1.0
4,4,"{""class"":""org.apache.spark.ml.classification.D...",1.0


In [10]:
TEST_MODEL_PREFIX = "artifacts/model/random_forest_model_test/"
ORIGINAL_MODEL_PREFIX = "artifacts/model/random_forest_model/"

all_blobs = list(container_client.list_blobs(name_starts_with=ORIGINAL_MODEL_PREFIX))
print(f"Tổng số file cần copy: {len(all_blobs)}")

for b in all_blobs:
    src_blob = container_client.get_blob_client(b.name)
    dst_name = b.name.replace(ORIGINAL_MODEL_PREFIX, TEST_MODEL_PREFIX, 1)
    dst_blob = container_client.get_blob_client(dst_name)
    dst_blob.start_copy_from_url(src_blob.url)

print("Đã copy xong sang:", TEST_MODEL_PREFIX)

Tổng số file cần copy: 36
✅ Đã copy xong sang: artifacts/model/random_forest_model_test/


In [11]:
import time
time.sleep(5)  # đợi copy hoàn tất (async)

# verify số file khớp
test_blobs = list(container_client.list_blobs(name_starts_with=TEST_MODEL_PREFIX))
print(f"Số file ở path test: {len(test_blobs)} (kỳ vọng khớp với {len(all_blobs)})")

Số file ở path test: 36 (kỳ vọng khớp với 36)


In [12]:
test_treesmeta_blob_name = parquet_blob_name.replace(ORIGINAL_MODEL_PREFIX, TEST_MODEL_PREFIX, 1)
print("Sẽ ghi đè file:", test_treesmeta_blob_name)

with open(fixed_local_path, "rb") as f:
    container_client.upload_blob(
        name=test_treesmeta_blob_name,
        data=f,
        overwrite=True
    )
print("Đã thay file fix vào bản test")

Sẽ ghi đè file: artifacts/model/random_forest_model_test/stages/3_RandomForestClassifier_b44ed63ef3d2/treesMetadata/part-00000-tid-4148136318540938165-0e677ae4-154e-4e4b-8b1b-2aeb2b8b885e-729-1.c000.snappy.parquet
✅ Đã thay file fix vào bản test


In [13]:
from pyspark.ml import PipelineModel

TEST_MODEL_PATH = f"{BASE_PATH}/{TEST_MODEL_PREFIX}"
try:
    model_test = PipelineModel.load(TEST_MODEL_PATH)
    print("Load thành công từ bản test")
    rf_model = model_test.stages[-1]
    print("numTrees:", rf_model.getNumTrees)
except Exception as e:
    print("Vẫn lỗi:")
    print(repr(e))

✅ Load thành công từ bản test
numTrees: 80


In [14]:
TEST_FEATURES_PATH = f"{BASE_PATH}/gold/training_features"

# lấy 1 tháng gần nhất trong Test set (2026) để thử, theo đúng partition filter như tài liệu mô tả
df_sample = (spark.read.parquet(TEST_FEATURES_PATH)
             .filter("year = 2026")
             .limit(5))

df_sample.show(truncate=80)
print("Số cột:", len(df_sample.columns))
df_sample.printSchema()

+-------+-------+----+-----------+----------+----------+-------------------+------------------+---------------------+----------------------+---------------------+----------------------+---------------------+----------------------+---------------------+----------------------+----------------+--------------------+-----------------------+-------------------+----+-----+
|link_id|borough|hour|day_of_week|is_weekend|is_holiday|        speed_ratio|current_congestion|past_congestion_15min|past_speed_ratio_15min|past_congestion_30min|past_speed_ratio_30min|past_congestion_45min|past_speed_ratio_45min|past_congestion_60min|past_speed_ratio_60min|congestion_trend|   speed_ratio_trend|future_congestion_15min|         data_as_of|year|month|
+-------+-------+----+-----------+----------+----------+-------------------+------------------+---------------------+----------------------+---------------------+----------------------+---------------------+----------------------+---------------------+----------

In [15]:
predictions = model_test.transform(df_sample)

predictions.select(
    "link_id", "borough",
    "current_congestion", "future_congestion_15min",  # label thật để so sánh
    "prediction", "probability"
).show(truncate=80)

+-------+-------+------------------+-----------------------+----------+------------------------------------------------------------+
|link_id|borough|current_congestion|future_congestion_15min|prediction|                                                 probability|
+-------+-------+------------------+-----------------------+----------+------------------------------------------------------------+
|4616355| Queens|                 1|                      1|       1.0| [0.21975322978225184,0.4823135192386063,0.2979332509791419]|
|4620332|  Bronx|                 0|                      0|       0.0|[0.8118043231772152,0.15094223376996865,0.03725344305281606]|
|4616355| Queens|                 1|                      1|       2.0|[0.18024399301667746,0.36343227452347915,0.4563237324598434]|
|4620332|  Bronx|                 0|                      0|       0.0|[0.8589587530067517,0.10644843827957291,0.03459280871367537]|
|4616355| Queens|                 1|                      1|       1.

In [16]:
with open(fixed_local_path, "rb") as f:
    container_client.upload_blob(
        name=parquet_blob_name,   # path gốc lấy từ cell đầu tiên
        data=f,
        overwrite=True
    )
print("Đã ghi đè file fix vào bản gốc:", parquet_blob_name)

Đã ghi đè file fix vào bản gốc: artifacts/model/random_forest_model/stages/3_RandomForestClassifier_b44ed63ef3d2/treesMetadata/part-00000-tid-4148136318540938165-0e677ae4-154e-4e4b-8b1b-2aeb2b8b885e-729-1.c000.snappy.parquet


In [17]:
MODEL_PATH = f"{BASE_PATH}/artifacts/model/random_forest_model"
model_final = PipelineModel.load(MODEL_PATH)
print("Model gốc load OK, numTrees:", model_final.stages[-1].getNumTrees)

Model gốc load OK, numTrees: 80


In [18]:
test_blobs = list(container_client.list_blobs(name_starts_with=TEST_MODEL_PREFIX))
for b in test_blobs:
    container_client.delete_blob(b.name)
print(f"Đã xóa {len(test_blobs)} file ở thư mục test")

Đã xóa 36 file ở thư mục test


In [3]:
prefix = "artifacts/model/model_test/"
blobs = list(container_client.list_blobs(name_starts_with=prefix))
for b in blobs:
    print(b.name)

artifacts/model/model_test/metadata
artifacts/model/model_test/metadata/_SUCCESS
artifacts/model/model_test/metadata/_committed_7168792348750245639
artifacts/model/model_test/metadata/_started_7168792348750245639
artifacts/model/model_test/metadata/part-00000-tid-7168792348750245639-04d02605-c890-4b45-80df-627fca43af4d-259-1-c000.txt
artifacts/model/model_test/stages
artifacts/model/model_test/stages/0_StringIndexer_2df7e607c920
artifacts/model/model_test/stages/0_StringIndexer_2df7e607c920/data
artifacts/model/model_test/stages/0_StringIndexer_2df7e607c920/data/_SUCCESS
artifacts/model/model_test/stages/0_StringIndexer_2df7e607c920/data/_committed_3558493274448829984
artifacts/model/model_test/stages/0_StringIndexer_2df7e607c920/data/_started_3558493274448829984
artifacts/model/model_test/stages/0_StringIndexer_2df7e607c920/data/part-00000-tid-3558493274448829984-f11acf56-6993-4822-8ec1-aabf9f6a0fd6-261-1.c000.snappy.parquet
artifacts/model/model_test/stages/0_StringIndexer_2df7e607c9

In [5]:
from pyspark.ml import PipelineModel
import traceback

TEST_MODEL_PATH = f"{BASE_PATH}/{prefix}"

try:
    model_test = PipelineModel.load(TEST_MODEL_PATH)
    print("Load thành công từ bản test")
    rf_model = model_test.stages[-1]
    print("numTrees:", rf_model.getNumTrees)
except Exception as e:
    print("Vẫn lỗi:")
    traceback.print_exc()

Load thành công từ bản test
numTrees: 80


In [6]:
TEST_FEATURES_PATH = f"{BASE_PATH}/gold/training_features"

# lấy 1 tháng gần nhất trong Test set (2026) để thử, theo đúng partition filter như tài liệu mô tả
df_sample = (spark.read.parquet(TEST_FEATURES_PATH)
             .filter("year = 2026")
             .limit(5))

df_sample.show(truncate=80)
print("Số cột:", len(df_sample.columns))
df_sample.printSchema()

+-------+-------+----+-----------+----------+----------+-------------------+------------------+---------------------+----------------------+---------------------+----------------------+---------------------+----------------------+---------------------+----------------------+----------------+--------------------+-----------------------+-------------------+----+-----+
|link_id|borough|hour|day_of_week|is_weekend|is_holiday|        speed_ratio|current_congestion|past_congestion_15min|past_speed_ratio_15min|past_congestion_30min|past_speed_ratio_30min|past_congestion_45min|past_speed_ratio_45min|past_congestion_60min|past_speed_ratio_60min|congestion_trend|   speed_ratio_trend|future_congestion_15min|         data_as_of|year|month|
+-------+-------+----+-----------+----------+----------+-------------------+------------------+---------------------+----------------------+---------------------+----------------------+---------------------+----------------------+---------------------+----------

In [7]:
predictions = model_test.transform(df_sample)

predictions.select(
    "link_id", "borough",
    "current_congestion", "future_congestion_15min",  # label thật để so sánh
    "prediction", "probability"
).show(truncate=80)

+-------+-------+------------------+-----------------------+----------+-------------------------------------------------------------+
|link_id|borough|current_congestion|future_congestion_15min|prediction|                                                  probability|
+-------+-------+------------------+-----------------------+----------+-------------------------------------------------------------+
|4616355| Queens|                 1|                      1|       1.0| [0.26527843305660637,0.4737003441777087,0.26102122276568496]|
|4620332|  Bronx|                 0|                      0|       0.0| [0.8142190252139416,0.14602297952730967,0.03975799525874865]|
|4616355| Queens|                 1|                      1|       2.0|  [0.18102828032713217,0.36953664160579675,0.449435078067071]|
|4620332|  Bronx|                 0|                      0|       0.0|[0.8677444238429602,0.10178473737023921,0.030470838786800503]|
|4616355| Queens|                 1|                      1|  